In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/multiclassificationtask/sample_submission.csv
/kaggle/input/competitions/multiclassificationtask/train.csv
/kaggle/input/competitions/multiclassificationtask/test.csv


In [4]:
df = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/train.csv')

print(df.shape)
print(df.head())
print(df.dtypes)


(15000, 20)
   id  N_Days             Drug      Age Sex Ascites Hepatomegaly Spiders  \
0   0  2178.0  D-penicillamine  16374.0   F       N            N       N   
1   1  2644.0  D-penicillamine  17774.0   F       N            N       N   
2   2  3069.0          Placebo  17844.0   F       N            N       N   
3   3  2216.0          Placebo  19221.0   F       N            Y       Y   
4   4  2256.0          Placebo  21600.0   F       N            N       N   

  Edema  Bilirubin  Cholesterol  Albumin  Copper  Alk_Phos    SGOT  \
0     N        0.5        263.0     3.20    43.0    1110.0  106.95   
1     N        0.8        280.0     3.60    22.0     678.0   62.00   
2     N        1.1        408.0     4.40    54.0    2108.0  142.60   
3     N        0.8        252.0     3.70    36.0     843.0   55.80   
4     N        4.7        348.0     3.06   464.0     961.0  120.90   

   Tryglicerides  Platelets  Prothrombin  Stage Status  
0           67.0      430.0          9.6    3.0      

In [5]:
print(df.isnull().sum())
print(df['Status'].value_counts())
print(df['Edema'].unique())

id                  0
N_Days              0
Drug             6506
Age                 0
Sex                 0
Ascites          6498
Hepatomegaly     6508
Spiders          6509
Edema               0
Bilirubin           0
Cholesterol      8299
Albumin             0
Copper           6601
Alk_Phos         6512
SGOT             6514
Tryglicerides    8334
Platelets         564
Prothrombin        16
Stage               0
Status              0
dtype: int64
Status
C     10053
D      4565
CL      381
Y         1
Name: count, dtype: int64
['N' 'S' 'Y']


In [6]:
print(df.groupby('Status')['Cholesterol'].apply(lambda x: x.isnull().mean()))

Status
C     0.544614
CL    0.475066
D     0.578970
Y     0.000000
Name: Cholesterol, dtype: float64


In [7]:
#df['Cholesterol'].fillna(df.groupby('Stage')['Cholesterol'].transform('median'))
df = df[df['Status'] != 'Y']
print(df['Status'].value_counts())


Status
C     10053
D      4565
CL      381
Name: count, dtype: int64


In [8]:
df['Age']=df['Age']  / 365
print(df['Age'].describe())

count    14999.000000
mean        52.872807
std         10.404638
min          1.095890
25%         45.638356
50%         53.545205
75%         61.224658
max        354.515068
Name: Age, dtype: float64


In [9]:
df[(df['Age'] <= 18) | (df['Age'] >=100)]

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
489,489,1150.0,NaN,3.150685,F,NaN,NaN,NaN,N,1.6,NaN,3.00,NaN,NaN,NaN,NaN,NaN,10.1,2.0,C
1101,1101,1223.0,NaN,3.350685,F,NaN,NaN,NaN,N,1.2,NaN,3.50,NaN,NaN,NaN,NaN,167.0,10.4,2.0,C
2780,2780,3445.0,NaN,9.501370,F,NaN,NaN,NaN,N,0.6,NaN,3.83,NaN,NaN,NaN,NaN,273.0,10.8,2.0,C
3350,3350,1067.0,D-penicillamine,354.515068,F,N,N,N,N,0.8,242.0,4.01,20.0,622.0,66.65,88.0,358.0,11.0,3.0,C
5031,5031,2995.0,Placebo,8.205479,F,N,N,N,N,1.1,236.0,3.52,38.0,791.0,57.35,114.0,32.0,9.9,3.0,C
5828,5828,2249.0,D-penicillamine,1.095890,F,N,N,N,N,0.6,NaN,3.70,22.0,858.0,55.80,NaN,224.0,10.6,1.0,C
6220,6220,2171.0,D-penicillamine,5.569863,F,N,N,N,N,0.6,NaN,3.85,20.0,674.0,57.35,NaN,146.0,9.9,3.0,C
6842,6842,3445.0,NaN,9.501370,F,NaN,NaN,NaN,N,0.7,NaN,3.93,NaN,NaN,NaN,NaN,378.0,11.1,2.0,C
6904,6904,1223.0,NaN,3.350685,F,NaN,NaN,NaN,N,0.9,NaN,3.00,NaN,NaN,NaN,NaN,224.0,10.5,2.0,C
9449,9449,673.0,D-penicillamine,255.632877,F,N,N,N,N,0.8,340.0,3.61,38.0,1828.0,71.30,93.0,390.0,10.6,3.0,D


In [10]:
print(len(df[(df['Age'] < 18) | (df['Age'] > 100)]))

14


## Baseline pipeline (CV + submission)
Bu bo'limda train/test ni qayta yuklaymiz, oddiy va reproduktiv pipeline quramiz, log loss ni baholaymiz va `submission.csv` yaratamiz.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
N_SPLITS = 5

In [ ]:
# Kaggle notebook path
TRAIN_PATH = '/kaggle/input/competitions/multiclassificationtask/train.csv'
TEST_PATH = '/kaggle/input/competitions/multiclassificationtask/test.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Target distribution (raw):')
print(train_df['Status'].value_counts(dropna=False))

In [ ]:
# Competition needs 3 classes only: C, CL, D
train_df = train_df[train_df['Status'].isin(['C', 'CL', 'D'])].copy()

X = train_df.drop(columns=['Status'])
y = train_df['Status']
X_test = test_df.copy()

# Optional feature engineering copied from exploration
X['Age_years'] = X['Age'] / 365.0
X_test['Age_years'] = X_test['Age'] / 365.0

# Keep original id for submission
test_ids = X_test['id'].copy()

# We don't want id to influence the model directly
X = X.drop(columns=['id'])
X_test = X_test.drop(columns=['id'])

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = [c for c in X.columns if c not in categorical_cols]

print('Categorical columns:', len(categorical_cols))
print('Numerical columns:', len(numerical_cols))

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=RANDOM_STATE)),
    ]
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_pred = np.zeros((len(X), y.nunique()))
label_order = sorted(y.unique())

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model.fit(X_train, y_train)
    pred_valid = model.predict_proba(X_valid)

    # Align columns by classifier classes_
    class_to_idx = {c: i for i, c in enumerate(model.named_steps['clf'].classes_)}
    aligned = np.column_stack([pred_valid[:, class_to_idx[c]] for c in label_order])
    oof_pred[valid_idx, :] = aligned

    fold_loss = log_loss(y_valid, aligned, labels=label_order)
    print(f'Fold {fold} log loss: {fold_loss:.6f}')

overall_loss = log_loss(y, oof_pred, labels=label_order)
print(f'\nOOF log loss: {overall_loss:.6f}')
print('Label order used:', label_order)

In [ ]:
# Train on full data and create submission
model.fit(X, y)
test_pred = model.predict_proba(X_test)

class_to_idx_full = {c: i for i, c in enumerate(model.named_steps['clf'].classes_)}

submission = pd.DataFrame({
    'id': test_ids,
    'Status_C': test_pred[:, class_to_idx_full['C']],
    'Status_CL': test_pred[:, class_to_idx_full['CL']],
    'Status_D': test_pred[:, class_to_idx_full['D']],
})

# Safety normalization (row-wise) for stable probabilities
prob_cols = ['Status_C', 'Status_CL', 'Status_D']
submission[prob_cols] = submission[prob_cols].div(submission[prob_cols].sum(axis=1), axis=0)

submission.to_csv('/kaggle/working/submission.csv', index=False)
submission.head()